<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/ecomarket-solution/blob/main/notebooks/EcoMarket_AI_Solution.ipynb)


# Maestría en Inteligencia Artificial  
## IA Generativa
---
# EcoMarket AI Support — Taller Práctico #2

**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  

## Optimización de Atención al Cliente con IA Generativa (RAG) para asistente de e-commerce

En este notebook se desarrolla un chatbot inteligente orientado a responder consultas de clientes en un entorno de comercio electrónico, específicamente sobre estado de pedidos y procesos de devolución.

A diferencia de los enfoques tradicionales de generación de texto, este sistema utiliza la técnica de Retrieval-Augmented Generation (RAG), que combina la recuperación de información con modelos generativos de lenguaje.

El modelo no se limita a generar respuestas basadas únicamente en su conocimiento previo. En su lugar, primero recupera información relevante desde una base de conocimiento que incluye datos de pedidos y políticas de la tienda. Posteriormente, utiliza este contexto para generar respuestas más precisas, coherentes y alineadas con la información real del negocio.



Configura rutas y detecta si el entorno es Colab o local.

In [1]:
import warnings
from importlib import metadata

warnings.filterwarnings('ignore')

installed_packages = {dist.metadata['Name'].lower() for dist in metadata.distributions() if dist.metadata.get('Name')}
IN_COLAB = 'google-colab' in installed_packages

Descarga dependencias y las instala automáticamente en Colab.

In [2]:
if IN_COLAB:
    !wget -q https://raw.githubusercontent.com/sebastianb92/ecomarket-rag-solution/main/requirements.txt -O requirements.txt && \
    uv pip install -r requirements.txt
else:
    print("Entorno local — instala con: uv pip install -r requirements.txt")

Using Python 3.12.13 environment at: /usr
Resolved 193 packages in 3.35s
Prepared 31 packages in 1.15s
Uninstalled 2 packages in 90ms
Installed 31 packages in 94ms
 + bcrypt==5.0.0
 + build==1.4.4
 + chromadb==1.5.8
 + colab-xterm==0.2.0
 + dataclasses-json==0.6.7
 + durationpy==0.10
 + filetype==1.2.0
 + groq==0.37.1
 + jq==1.11.0
 + kubernetes==35.0.0
 + langchain-chroma==1.1.0
 + langchain-classic==1.0.4
 + langchain-community==0.4.1
 - langchain-core==1.2.28
 + langchain-core==1.3.0
 + langchain-google-genai==4.2.2
 + langchain-groq==1.1.2
 + langchain-huggingface==1.2.2
 + langchain-ollama==1.1.0
 + langchain-openai==1.2.0
 + langchain-text-splitters==1.1.2
 + marshmallow==3.26.2
 + mypy-extensions==1.1.0
 + ollama==0.6.1
 + onnxruntime==1.25.0
 + opentelemetry-exporter-otlp-proto-grpc==1.38.0
 + pybase64==1.4.3
 + pypdf==6.10.2
 + pypika==0.51.1
 + pyproject-hooks==1.2.0
 - requests==2.32.4
 + requests==2.33.1
 + typing-inspect==0.9.0


IMPORTACIONES

In [3]:
# Librerías estándar
import os
from pathlib import Path

# Datos
import pandas as pd

# Entorno
from google.colab import userdata

# Modelos
from langchain_groq import ChatGroq

# Embeddings y vector store
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata

# Carga de documentos
from langchain_community.document_loaders import (
    DataFrameLoader,
    PyPDFLoader,
    JSONLoader
)

# Procesamiento de texto
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Prompts y cadenas
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import RetrievalQA

Descarga los datos necesarios desde GitHub en Colab si no existen; en local usa los archivos ya disponibles.

In [4]:
if IN_COLAB:
    !mkdir -p data

    # JSON
    ![ ! -f data/FAQ.json ] && wget -q -O data/FAQ.json \
    https://raw.githubusercontent.com/sebastianb92/ecomarket-rag-solution/main/data/FAQ.json

    # PDF
    ![ ! -f data/politica_devoluciones.pdf ] && wget -q -O data/politica_devoluciones.pdf \
    "https://raw.githubusercontent.com/sebastianb92/ecomarket-rag-solution/main/data/POL%C3%8DTICA%20DE%20DEVOLUCIONES.pdf"

    # Excel
    ![ ! -f data/pedidos_ecomarket.xlsx ] && wget -q -O data/pedidos_ecomarket.xlsx \
    https://raw.githubusercontent.com/sebastianb92/ecomarket-rag-solution/main/data/pedidos_ecomarket.xlsx

    print("Datos descargados desde GitHub")

else:
    print(f"Datos locales en: {DATA_DIR}")

Datos descargados desde GitHub


## Implementación de RAG

En esta sección se construye el pipeline completo de Retrieval-Augmented Generation (RAG), integrando los diferentes componentes necesarios para recuperar información relevante de EcoMarket y generar respuestas en lenguaje natural.

### Componentes del sistema

1. **LLM (GROQ):**  `llama-3.3-70b-versatile`

   Modelo generativo encargado de producir la respuesta final en lenguaje natural, utilizando como contexto los documentos recuperados.

2. **Embeddings:** `intfloat/multilingual-e5-large`  
   Modelo avanzado de Hugging Face diseñado para tareas de recuperación semántica. Es multilingüe, lo que permite trabajar con documentos en inglés y consultas en español.  Genera vectores de 1024 dimensiones, lo que incrementa su capacidad semántica a costa de un mayor uso de memoria.
   
3. **Document Store (Vector Store):**  
   Se utiliza CHROMA (a través de LangChain), una librería optimizada para la indexación y búsqueda eficiente de vectores en espacios de alta dimensión.


Configura el modelo **LLM** de Groq con la API key y parámetros de generación.

In [5]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=userdata.get("GROQ_API_KEY"),
    temperature=0.3 #Respuestas más deterministas (menos creativas)
)

Inicializa el modelo de **embeddings** para representar texto como vectores.

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={"device": "cuda"},  # cpu o "cuda"
    encode_kwargs={"normalize_embeddings": True}
)

/tmp/ipykernel_704/2589624455.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]


Crea y configura la **vector store** para almacenar y consultar embeddings.

In [7]:
vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",
)

## Indexación de datos

En esta sección se prepara la información para que pueda ser buscada eficientemente (base de un sistema RAG).

El proceso de indexación sigue tres pasos principales:

**Load (Carga):** Se cargan los datos desde diferentes fuentes como JSON, PDF o Excel usando loaders.

**Split (Fragmentación):** Los documentos se dividen en fragmentos más pequeños para facilitar su procesamiento y mejorar la búsqueda.

**Store (Almacenamiento):** Los fragmentos se convierten en embeddings y se almacenan en una base vectorial para permitir búsquedas semánticas posteriores.

Fuente: https://docs.langchain.com/oss/python/langchain/rag#huggingface

<img src="https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_indexing.png?w=840&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=1838328a870c7353c42bf1cc2290a779" width="900">

Carga un archivo Excel, convierte cada fila en texto y la transforma en documentos para el sistema RAG.

In [8]:
df = pd.read_excel("/content/data/pedidos_ecomarket.xlsx")

df["contenido"] = df.astype(str).agg(" | ".join, axis=1)

loader = DataFrameLoader(df, page_content_column="contenido")
docs_excel = loader.load()

Carga el archivo PDF y lo convierte en documentos procesables para el RAG.

In [9]:
loader = PyPDFLoader("/content/data/politica_devoluciones.pdf")
docs_pdf = loader.load()

Carga el archivo JSON y transforma cada registro en texto estructurado para el RAG.

In [10]:
loader = JSONLoader(
    file_path="/content/data/FAQ.json",
    jq_schema='.faq[] | "Categoría: \(.categoria)\nPregunta: \(.pregunta)\nRespuesta: \(.respuesta)"',
    text_content=True
)

docs_json = loader.load()

Une todos los documentos en una sola colección para su procesamiento.

In [11]:
docs = docs_excel + docs_pdf + docs_json

Divide los documentos en fragmentos más pequeños para mejorar la búsqueda y el procesamiento.

In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Se dividieron los documentos en {len(all_splits)} sub-documentos.")

Se dividieron los documentos en 55 sub-documentos.


Limpia metadatos complejos de los documentos para evitar errores al almacenarlos.

In [13]:
all_splits = filter_complex_metadata(all_splits)

Agrega los documentos a la base vectorial y muestra algunos IDs generados.

In [14]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['2fc6046b-e9c9-43fe-85b9-de0da2f06354', '35a2db7f-69ac-492d-9745-23ad4e68a191', '44070b2e-daa2-4c54-ae40-1e8558d5ec88']


## Recuperación y generación

En esta etapa se construye la lógica de respuesta del sistema RAG.

**Retrieve (Recuperación):** A partir de la pregunta del usuario, se buscan los fragmentos más relevantes en la base vectorial usando un retriever.

**Generate (Generación):** Un modelo genera la respuesta utilizando un prompt que combina la pregunta con la información recuperada.

Fuente: https://docs.langchain.com/oss/python/langchain/rag#huggingface

<img src="https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_retrieval_generation.png?w=840&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=67fe2302e241fc24238a5df1cf56573d" width="900">

Configura el **retriever** para buscar los 3 documentos más similares en la base vectorial.

In [15]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

Este prompt define a EcoBot, un asistente de e-commerce que responde consultas sobre estado de pedidos y devoluciones usando información de un contexto (RAG).

Primero clasifica la intención del usuario y luego aplica un flujo específico. Incluye una regla crítica: solo usa información de pedidos si el usuario proporciona explícitamente un número de pedido, evitando errores comunes de asociación automática entre productos y pedidos.


In [16]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres EcoBot, asistente de e-commerce.

PERSONALIDAD: Amable, honesto, empático y proactivo.

OBJETIVO:
Resolver consultas sobre:
1) Estado de pedidos
2) Devoluciones

REGLAS GENERALES:
- Usa SOLO el contexto proporcionado
- NO inventes información
- Si no encuentras datos, dilo claramente
- Responde SIEMPRE en español
- Tono amable, claro y profesional
- Sé conciso pero útil

--------------------------------------------------
PASO 0 — IDENTIFICAR INTENCIÓN
--------------------------------------------------
Clasifica la pregunta del usuario en UNA de estas categorías:

A) ESTADO_PEDIDO → si pregunta por envío, entrega, tracking, estado
B) DEVOLUCION → si pregunta por devolver, reembolso, cambios
C) AMBIGUO → si no está claro

- Si es AMBIGUO → pide aclaración breve antes de continuar

--------------------------------------------------
REGLA DE PRIORIDAD (CRÍTICA - OBLIGATORIA)
--------------------------------------------------

Cuando la consulta sea sobre DEVOLUCIONES:

- SOLO puedes usar un pedido si el usuario menciona explícitamente un número de pedido
  (ej: ECO-12345, pedido 123, etc.)

- PROHIBIDO:
  - Usar pedidos encontrados en el contexto si el usuario NO los mencionó
  - Inferir pedidos a partir del producto
  - Relacionar automáticamente producto ↔ pedido

- Si el usuario menciona SOLO un producto (ej: "set de cubiertos de bambú"):
  → IGNORA completamente cualquier pedido en el contexto
  → Responde como CONSULTA GENERAL (B2)

Esta regla tiene prioridad sobre cualquier otra instrucción.

--------------------------------------------------
FLUJO A — ESTADO DEL PEDIDO
--------------------------------------------------

PASO 1 — Buscar número de pedido en el contexto

PASO 2 — RESPUESTA

SI el pedido EXISTE:
- Explica el estado claramente

FORMATO OBLIGATORIO:

Hola [Nombre si está disponible, si no usa "Hola"],

Aquí tienes la información de tu pedido:

- Estado:
- Producto:
- Fecha estimada:
- Tracking:
- Última ubicación:
- Observaciones:

REGLAS ADICIONALES:
- Si está RETRASADO:
  → Empieza con disculpa sincera
  → Explica causa + nueva fecha

- Si está CANCELADO:
  → Explica estado del reembolso + plazo claro

- Si está PENDIENTE DE PAGO:
  → Indica acción requerida con tono amable

- Siempre indica el siguiente paso esperado

---

SI el pedido NO EXISTE:
- Indica que no se encontró el pedido
- Sugiere verificar el número
- Ofrece contacto:
  soporte@ecomarket.com
  900-ECO-MKT
- NUNCA inventes información

--------------------------------------------------
FLUJO B — DEVOLUCIONES
--------------------------------------------------

PASO 1 — IDENTIFICAR TIPO DE SOLICITUD

Clasifica en:

B1) DEVOLUCIÓN CON PEDIDO:
- SOLO si el usuario proporciona explícitamente un número de pedido

B2) CONSULTA GENERAL DE DEVOLUCIÓN:
- El usuario NO proporciona número de pedido
- Solo menciona un producto
- Pregunta por proceso, política o condiciones

--------------------------------------------------
CASO B1 — DEVOLUCIÓN CON PEDIDO
--------------------------------------------------

PASO 2 — Evaluar si aplica política usando el contexto

SI APLICA:

Hola [Nombre],

Claro, puedo ayudarte con la devolución de tu producto.

Sigue estos pasos:

1. [Paso 1]
2. [Paso 2]
3. [Paso 3]

- Plazo de reembolso:
- Método:

Puedes optar por crédito en tienda con 5% adicional.

---

SI NO APLICA:

Hola [Nombre],

1. Entiendo cómo te sientes con esta situación.
2. [Razón clara y humana]
3. [Alternativa concreta]

- Nunca respondas con un "no" sin alternativa

--------------------------------------------------
CASO B2 — CONSULTA GENERAL (SIN PEDIDO)
--------------------------------------------------

NO usar información de pedidos aunque exista en el contexto

FORMATO:

Hola,

Con gusto te explico cómo funciona nuestro proceso de devoluciones:

[Explicación clara basada SOLO en políticas del contexto]

Si deseas iniciar una devolución, necesitarás tu número de pedido.

¿Tienes el número de pedido? Con eso puedo ayudarte mejor.

--------------------------------------------------
CIERRE (SIEMPRE)
--------------------------------------------------

¿Puedo ayudarte con algo más?
"""),

    ("human", """Contexto:
{context}

Pregunta:
{question}
""")
])

Crea una cadena RAG que combina el modelo y el retriever para **generar** respuestas.

In [17]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt}
)

---
## Ejercicio 1 — Consultas de Estado de Pedido
Demuestra el manejo de diferentes estados del producto y la protección anti-alucinación.

Pedido EN TRÁNSITO

In [18]:
query = "¿Quiero saber el estado de mi pedido ECO-12345?"
respuesta = qa_chain.run(query)

print(respuesta)

/tmp/ipykernel_704/3993197727.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  respuesta = qa_chain.run(query)


Hola María,

Aquí tienes la información de tu pedido:

- Estado: En tránsito
- Producto: Kit de bambú reutilizable (3 piezas)
- Fecha estimada: 2024-07-05
- Tracking: https://track.dhl.com/ECO12345
- Última ubicación: Centro logístico Madrid
- Observaciones: Sin retrasos reportados.

Puedes seguir el estado de tu pedido a través del enlace de tracking proporcionado. Si necesitas más información o tienes alguna inquietud, no dudes en contactarnos.

¿Puedo ayudarte con algo más?


Pedido RETRASADO

In [19]:
query = "¿Quiero saber el estado de mi pedido ECO-12346?"
respuesta = qa_chain.run(query)

print(respuesta)

Hola Carlos,

Aquí tienes la información de tu pedido:

- Estado: RETRASADO
- Producto: Jabón orgánico artesanal (pack x6)
- Fecha estimada: 2024-07-08
- Tracking: https://track.correos.es/ECO12346
- Última ubicación: No disponible
- Observaciones: El retraso se debe a la huelga de transporte regional en Cataluña.

Lamentamos el retraso en tu pedido y queremos asegurarte que estamos trabajando para resolver el problema lo antes posible. Te pedimos disculpas por cualquier inconveniente que esto te pueda causar.

El siguiente paso es esperar a que se resuelva la huelga y se reactive el transporte. Te recomendamos revisar el tracking para obtener información actualizada sobre el estado de tu pedido.

¿Puedo ayudarte con algo más?


Pedido ENTREGADO

In [20]:
query = "¿Quiero saber el estado de mi pedido ECO-12347?"
respuesta = qa_chain.run(query)

print(respuesta)

Hola Ana,

Aquí tienes la información de tu pedido:

- Estado: ENTREGADO
- Producto: Botella de acero inoxidable 750ml
- Fecha estimada: 2024-06-25
- Tracking: https://track.seur.com/ECO12347
- Última ubicación: Entrega exitosa en dirección principal
- Observaciones: No hay observaciones adicionales.

El siguiente paso esperado es que disfrutes de tu producto. Si necesitas algo más, no dudes en contactarnos.

¿Puedo ayudarte con algo más?


Pedido CANCELADO

In [21]:
query = "¿Quiero saber el estado de mi pedido ECO-12349?"
respuesta = qa_chain.run(query)

print(respuesta)

Hola Laura,

Aquí tienes la información de tu pedido:

- Estado: CANCELADO
- Producto: Cepillo de dientes de bambú (pack x4)
- Fecha estimada: No aplica
- Tracking: No aplica
- Última ubicación: No aplica
- Observaciones: Cancelación procesada correctamente

El siguiente paso esperado es que recibas el reembolso correspondiente a tu pedido cancelado. Si tienes alguna pregunta adicional o necesitas más información, no dudes en preguntar.

¿Puedo ayudarte con algo más?


Pedido PENDIENTE DE PAGO

In [22]:
query = "¿Quiero saber el estado de mi pedido ECO-12371?"
respuesta = qa_chain.run(query)

print(respuesta)

Hola Mariana,

Aquí tienes la información de tu pedido:

- Estado: PENDIENTE DE PAGO
- Producto: Kit limpieza ecológica
- Fecha estimada: 2024-07-06
- Tracking: No disponible
- Última ubicación: No disponible
- Observaciones: Pago no completado

Para continuar con el proceso de tu pedido, es necesario que completes el pago. Te pedimos amablemente que verifiques tu cuenta y completes la transacción pendiente. Si necesitas ayuda o tienes alguna duda, no dudes en contactarnos.

¿Puedo ayudarte con algo más?


Número INEXISTENTE

In [23]:
query = "¿Quiero saber el estado de mi pedido ECO-99999?"
respuesta = qa_chain.run(query)

print(respuesta)

Hola,

Aquí tienes la información de tu pedido:

- Estado: Lo siento, no tengo acceso a información en tiempo real sobre el estado de tus pedidos. Sin embargo, puedo sugerirte algunas opciones para que puedas obtener la información que necesitas.
- Producto: No tengo información específica sobre el producto de tu pedido.
- Fecha estimada: No tengo información sobre la fecha estimada de entrega.
- Tracking: Puedes revisar el estado de tu pedido utilizando el enlace de seguimiento (tracking_url) que se te envió a tu correo electrónico.
- Última ubicación: No tengo información sobre la última ubicación de tu pedido.
- Observaciones: Si tu pedido está retrasado, te recomiendo contactar con nuestro soporte para obtener más información y asistencia.

El siguiente paso esperado es que revises el estado de tu pedido utilizando el enlace de seguimiento o que contactes con nuestro soporte para obtener más información.

¿Puedo ayudarte con algo más?


## Ejercicio 2 — Solicitudes de Devolución
El desafío: responder con empatía tanto el «sí» como el «no»,
y nunca dar una negativa sin ofrecer una alternativa.

Devolución APROBADA

In [24]:
query = "Quiero hacer la devolucion de mi Botella de acero inoxidable 750ml ya que el diseño no me gusta, ¿que debo hacer?"
respuesta = qa_chain.run(query)

print(respuesta)

Hola,

Con gusto te explico cómo funciona nuestro proceso de devoluciones:

Si deseas devolver un producto, puedes solicitar una devolución dentro de los 30 días naturales desde la fecha de entrega. 

Para iniciar el proceso de devolución, necesitarás proporcionar tu número de pedido. ¿Tienes el número de pedido? Con eso puedo ayudarte mejor.

Recuerda que, según nuestra política, puedes solicitar reembolso o reemplazo sin coste adicional si el producto está dañado, pero en este caso, mencionas que el diseño no te gusta. Te recomiendo verificar las condiciones de devolución para asegurarte de que cumple con nuestros requisitos.

¿Puedo ayudarte con algo más?


Devolución RECHAZADA

In [25]:
query = "Quiero hacer la devolucion de mi Jabón orgánico artesanal ya que no me gusta el olor, ¿que debo hacer?"
respuesta = qa_chain.run(query)

print(respuesta)

Hola,

Entiendo que deseas hacer la devolución de tu Jabón orgánico artesanal debido a que no te gusta el olor. Sin embargo, según nuestras políticas, los productos de higiene personal abiertos no son elegibles para devolución, a menos que presenten defectos o daños.

En este caso, como el motivo de la devolución es el olor, no se aplica a nuestra política de devolución. Sin embargo, te ofrezco una alternativa: si deseas, puedes contactar con nuestro equipo de atención al cliente para discutir posibles opciones, como un crédito en tienda o un intercambio por otro producto.

Si deseas proceder con la solicitud, te recomiendo revisar nuestra política de devolución en ecomarket.com/devoluciones para obtener más información.

¿Puedo ayudarte con algo más?


Devolución RECHAZADA

In [26]:
query = "Quiero hacer la devolucion del Mix de frutos secos orgánicos ya que cambie de opinion y no lo quiero, ¿que debo hacer?"
respuesta = qa_chain.run(query)

print(respuesta)

Hola,

Entiendo que deseas hacer la devolución del Mix de frutos secos orgánicos. Sin embargo, según nuestra política de devoluciones, los alimentos y productos perecederos, incluyendo los alimentos orgánicos, no son elegibles para devolución debido a restricciones de seguridad alimentaria e higiene.

En este caso, no podemos aceptar la devolución del producto. Si tienes alguna otra pregunta o inquietud, no dudes en hacérmelo saber.

¿Puedo ayudarte con algo más?


Devolución APROBADA

In [27]:
query = "Quiero hacer la devolucion del Cepillo de dientes de bambú ya que el paquete llegó aplastado y roto, ¿que debo hacer?"
respuesta = qa_chain.run(query)

print(respuesta)

Hola,

Con gusto te explico cómo funciona nuestro proceso de devoluciones:

Si has recibido un producto dañado, como en tu caso, el Cepillo de dientes de bambú que llegó aplastado y roto, debes reportarlo. Una vez reportado, podrás solicitar reembolso o reemplazo sin coste adicional.

Para proceder, necesitarás proporcionarme el número de pedido asociado a este producto. ¿Tienes el número de pedido? Con eso puedo ayudarte mejor a iniciar el proceso de devolución.

¿Puedo ayudarte con algo más?


Devolución FUERA DE PLAZO

In [28]:
query = "Quiero hacer la devolucion del Set de cubiertos de bambú ya que me regalaron otro, ¿que debo hacer?"
respuesta = qa_chain.run(query)

print(respuesta)

Hola,

Con gusto te explico cómo funciona nuestro proceso de devoluciones:

Si deseas devolver un producto, debes seguir nuestros pasos de devolución. Sin embargo, para poder ayudarte de manera más específica, necesitaré que me proporciones el número de pedido asociado al Set de cubiertos de bambú que deseas devolver.

¿Tienes el número de pedido? Con eso puedo ayudarte mejor. Recuerda que también puedes contactarnos en soporte@ecomarket.com o al 900-ECO-MKT para obtener asistencia adicional.

¿Puedo ayudarte con algo más?
